# Kaggle Pipeline: YOLO Then U-Net
This notebook builds the dataset generator, creates the MNISTDD-RGB dataset, then trains YOLO detection first and U-Net segmentation second (sequentially).
When available, all GPUs are used for each training stage.

In [ ]:
!nvidia-smi || true
!python3 --version

In [ ]:
!apt-get update -qq
!apt-get install -y -qq cmake ninja-build libopencv-dev
!python3 -m pip install -q ultralytics

In [ ]:
%cd /kaggle/working
!rm -rf Minor_Project
!git clone https://github.com/Yash4616/Minor_Project.git
%cd /kaggle/working/Minor_Project

In [ ]:
%cd /kaggle/working/Temp
!git pull

In [ ]:
!python3 scripts/download_mnist.py --out_dir /kaggle/working/data

In [ ]:
import torch
torch_prefix = torch.utils.cmake_prefix_path
print('Torch CMake prefix:', torch_prefix)
!rm -rf build
!cmake -S . -B build -G Ninja -DCMAKE_BUILD_TYPE=Release -DCMAKE_PREFIX_PATH="$torch_prefix"
!cmake --build build -j2

In [ ]:
!./build/Change --data_dir /kaggle/working/data --output_dir /kaggle/working/output --generate_only
!python3 scripts/train_yolo_then_unet.py --dataset_dir /kaggle/working/output/digit_dataset --output_dir /kaggle/working/output

In [ ]:
!ls -lah /kaggle/working/output
!find /kaggle/working/output -maxdepth 3 -type f | head -n 80
!sed -n '1,160p' /kaggle/working/output/report.txt